<a href="https://colab.research.google.com/github/Mazen-Morsy/ML-Final/blob/main/ml_assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# hello
# MAZEN 5WL W OMAR BYTASSSSSSSSSSSS

# Introduction to Machine Learning — Assignment 2

## 0. Imports and reproducibility

We use NumPy for the implementations, pandas for readable result tables, and matplotlib for plots. Scikit-learn is used only for loading the required datasets, splitting the data, and the explicit reference comparisons requested in the assignment.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, load_breast_cancer, load_digits, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    confusion_matrix as sklearn_confusion_matrix,
    precision_recall_fscore_support
)

np.set_printoptions(precision=6, suppress=True)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("NumPy version:", np.__version__)
print("All randomness in this notebook uses random_state / default_rng seed 42.")

## Part A — Data and toolkit

### Task 1. Load the four datasets and apply the required 60 / 20 / 20 split

The assignment specifies:
1. Split 20% into the test set using `random_state=42`.
2. Split 25% of the remaining 80% into validation (which is 20% of the full dataset).
3. Use stratification for classification datasets, but not for Diabetes regression.

Dataset-specific preparation:
- Diabetes: `load_diabetes(scaled=False)`; keep the target in its original units.
- Breast Cancer: flip the labels so malignant = 1 and benign = 0.
- Digits: divide pixel values by 16; do not standardize them.
- Iris: use the original four features and three classes.

In [ ]:
# Load datasets exactly as described in the assignment.
diabetes = load_diabetes(scaled=False)
breast = load_breast_cancer()
digits = load_digits()
iris = load_iris()

datasets = {}

# Diabetes: regression, so stratify=None.
datasets["Diabetes"] = {
    "X": diabetes.data.astype(float),
    "y": diabetes.target.astype(float),
    "feature_names": list(diabetes.feature_names),
    "task": "regression"
}

# Breast Cancer: malignant must be the positive class (1).
datasets["Breast cancer"] = {
    "X": breast.data.astype(float),
    "y": (1 - breast.target).astype(int),
    "feature_names": list(breast.feature_names),
    "task": "classification"
}

# Digits: scale pixel intensities to [0, 1] by dividing by 16.
datasets["Digits"] = {
    "X": digits.data.astype(float) / 16.0,
    "y": digits.target.astype(int),
    "feature_names": [f"pixel_{i}" for i in range(digits.data.shape[1])],
    "task": "classification"
}

datasets["Iris"] = {
    "X": iris.data.astype(float),
    "y": iris.target.astype(int),
    "feature_names": list(iris.feature_names),
    "task": "classification"
}

# Apply the exact two-step split protocol.
splits = {}
for name, data in datasets.items():
    X, y = data["X"], data["y"]
    stratify_all = y if data["task"] == "classification" else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=stratify_all
    )

    stratify_train = y_train if data["task"] == "classification" else None
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.25, random_state=42,
        stratify=stratify_train
    )

    splits[name] = {
        "X_train": X_train, "X_val": X_val, "X_test": X_test,
        "y_train": y_train, "y_val": y_val, "y_test": y_test,
        "feature_names": data["feature_names"],
        "task": data["task"]
    }

# Required split sizes.
expected_sizes = {
    "Diabetes": (264, 89, 89, 442),
    "Breast cancer": (341, 114, 114, 569),
    "Digits": (1077, 360, 360, 1797),
    "Iris": (90, 30, 30, 150)
}

rows = []
for name, sp in splits.items():
    sizes = tuple(len(sp[k]) for k in ["y_train", "y_val", "y_test"])
    total = sum(sizes)
    rows.append([name, *sizes, total, expected_sizes[name]])
    assert sizes + (total,) == expected_sizes[name], f"Unexpected split sizes for {name}"

split_table = pd.DataFrame(
    [[r[0], r[1], r[2], r[3], r[4]] for r in rows],
    columns=["Dataset", "Train", "Validation", "Test", "Total"]
)
display(split_table)
print("All split sizes match the assignment.")

mazen

## Part C — Logistic Regression

In this section, we implement binary Logistic Regression from scratch using NumPy. We train the model using gradient descent and evaluate it using the validation and test sets.

In [ ]:
# ==========================================
# PART C: LOGISTIC REGRESSION
# ==========================================

def sigmoid(z):
    """
    Converts any number into a probability between 0 and 1.

    We use sigmoid in Logistic Regression because we want
    to predict the probability that a sample belongs to class 1.
    """

    # Clip the values to prevent overflow when calculating exp()
    z = np.clip(z, -500, 500)

    # The sigmoid mathematical formula
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(y_true, y_prob):
    """
    Calculates the Binary Cross-Entropy loss.

    y_true: the actual labels (0 or 1)
    y_prob: the probabilities predicted by our model

    A smaller loss means our predictions are closer to the true labels.
    """

    # Small value to prevent log(0), which is undefined
    eps = 1e-12

    # Keep probabilities away from exactly 0 and 1
    y_prob = np.clip(y_prob, eps, 1 - eps)

    # Calculate the average Binary Cross-Entropy loss
    return -np.mean(
        y_true * np.log(y_prob) +
        (1 - y_true) * np.log(1 - y_prob)
    )


def logistic_loss(X, y, weights, reg_lambda=0.0):
    """
    Calculates the total Logistic Regression loss.

    X: input features, including the bias column
    y: actual labels
    weights: model parameters
    reg_lambda: controls the strength of L2 regularization
    """

    # Number of training samples
    n = len(y)

    # Calculate the predicted probabilities
    probabilities = sigmoid(X @ weights)

    # Calculate the normal Binary Cross-Entropy loss
    data_loss = binary_cross_entropy(y, probabilities)

    # Add L2 regularization to discourage very large weights.
    # weights[0] is the bias, so we do not regularize it.
    regularization = (
        reg_lambda / (2 * n)
    ) * np.sum(weights[1:] ** 2)

    # Return the data loss plus the regularization penalty
    return data_loss + regularization


def logistic_gradient(X, y, weights, reg_lambda=0.0):
    """
    Calculates the gradient of the Logistic Regression loss.

    The gradient tells us how each weight should change
    to reduce the loss.
    """

    # Number of samples
    n = len(y)

    # Get the predicted probabilities
    probabilities = sigmoid(X @ weights)

    # Calculate the gradient of the Binary Cross-Entropy loss
    gradient = (X.T @ (probabilities - y)) / n

    # Add the L2 regularization gradient to the feature weights.
    # We exclude the bias term from regularization.
    gradient[1:] += (reg_lambda / n) * weights[1:]

    return gradient


def train_logistic_regression(
    X, y, learning_rate=0.01,
    epochs=2000, reg_lambda=0.0
):
    """
    Trains Logistic Regression using Gradient Descent.

    learning_rate: controls how big each weight update is
    epochs: number of training iterations
    reg_lambda: regularization strength
    """

    # Get the number of samples and features
    n, d = X.shape

    # Add a column of ones to represent the bias parameter.
    # This allows the model to learn an intercept.
    X_bias = np.column_stack([np.ones(n), X])

    # Start all weights at zero
    weights = np.zeros(d + 1)

    # Store the loss after every epoch so we can plot it later
    losses = []

    # Repeat the training process
    for epoch in range(epochs):

        # Calculate the gradient using the current weights
        gradient = logistic_gradient(
            X_bias, y, weights, reg_lambda
        )

        # Update the weights in the direction that reduces the loss
        weights -= learning_rate * gradient

        # Calculate and save the current loss
        loss = logistic_loss(
            X_bias, y, weights, reg_lambda
        )

        losses.append(loss)

    # Return the learned weights and the loss history
    return weights, losses


def logistic_predict_proba(X, weights):
    """
    Returns the predicted probability for class 1.

    We add the bias column before multiplying by the weights.
    """

    X_bias = np.column_stack([np.ones(len(X)), X])

    # Calculate probabilities using the learned weights
    return sigmoid(X_bias @ weights)


def logistic_predict(X, weights, threshold=0.5):
    """
    Converts predicted probabilities into class labels.

    If probability >= 0.5, predict class 1.
    Otherwise, predict class 0.
    """

    probabilities = logistic_predict_proba(X, weights)

    return (probabilities >= threshold).astype(int)

In [ ]:
# Get the Breast Cancer dataset and its train/validation/test splits
bc = splits["Breast cancer"]

# Extract the features and labels for each split
X_train_bc = bc["X_train"]
X_val_bc = bc["X_val"]
X_test_bc = bc["X_test"]

y_train_bc = bc["y_train"]
y_val_bc = bc["y_val"]
y_test_bc = bc["y_test"]


# ==========================================
# FEATURE STANDARDIZATION
# ==========================================

# Calculate the mean of each feature using ONLY training data
bc_mean = X_train_bc.mean(axis=0)

# Calculate the standard deviation of each feature
bc_std = X_train_bc.std(axis=0)

# Prevent division by zero if a feature has no variation
bc_std[bc_std == 0] = 1

# Standardize the training features
X_train_bc = (X_train_bc - bc_mean) / bc_std

# Use the SAME training mean and standard deviation
# to standardize validation and test data.
# This prevents information leakage from those datasets.
X_val_bc = (X_val_bc - bc_mean) / bc_std
X_test_bc = (X_test_bc - bc_mean) / bc_std

In [ ]:
# ==========================================
# TRAIN LOGISTIC REGRESSION
# ==========================================

# Train the model using the training dataset
weights_bc, bc_losses = train_logistic_regression(
    X_train_bc,
    y_train_bc,
    learning_rate=0.01,  # Step size used by Gradient Descent
    epochs=2000,          # Number of training iterations
    reg_lambda=0.01       # L2 regularization strength
)


# ==========================================
# MAKE PREDICTIONS
# ==========================================

# Predict the labels for the training dataset
train_pred_bc = logistic_predict(X_train_bc, weights_bc)

# Predict the labels for the validation dataset
val_pred_bc = logistic_predict(X_val_bc, weights_bc)

# Predict the labels for the test dataset
test_pred_bc = logistic_predict(X_test_bc, weights_bc)


# ==========================================
# CALCULATE ACCURACY
# ==========================================

# Accuracy = number of correct predictions / total predictions

print("Training accuracy:",
      np.mean(train_pred_bc == y_train_bc))

print("Validation accuracy:",
      np.mean(val_pred_bc == y_val_bc))

print("Test accuracy:",
      np.mean(test_pred_bc == y_test_bc))


# ==========================================
# PLOT TRAINING LOSS
# ==========================================

plt.figure(figsize=(7, 4))

# Plot how the loss changes during training
plt.plot(bc_losses)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Logistic Regression Training Loss")

plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# PART D: SOFTMAX REGRESSION
# ==========================================

def softmax(logits):
    """
    Converts the model's raw scores (logits) into probabilities.

    Each row represents one sample.
    The probabilities in each row add up to 1.

    We subtract the maximum logit to make the calculation
    numerically stable and avoid excessively large exponentials.
    """

    shifted_logits = logits - np.max(
        logits, axis=1, keepdims=True
    )

    # Apply the exponential function to the shifted scores
    exp_values = np.exp(shifted_logits)

    # Divide each exponential by the sum of exponentials
    return exp_values / np.sum(
        exp_values, axis=1, keepdims=True
    )


def one_hot_encode(y, classes):
    """
    Converts class labels into one-hot encoded vectors.

    Example:
    Label 0 becomes [1, 0, 0]
    Label 1 becomes [0, 1, 0]
    Label 2 becomes [0, 0, 1]
    """

    # Compare every label with every possible class.
    # The result is 1 where they match and 0 otherwise.
    return (y[:, None] == classes[None, :]).astype(float)


def softmax_loss(X, y_onehot, weights, reg_lambda=0.0):
    """
    Calculates the Softmax Cross-Entropy loss.

    X: features including the bias column
    y_onehot: actual labels in one-hot format
    weights: model parameters for all classes
    reg_lambda: L2 regularization strength
    """

    # Number of training samples
    n = len(y_onehot)

    # Calculate predicted probabilities for every class
    probabilities = softmax(X @ weights)

    # Prevent log(0) when calculating cross-entropy
    eps = 1e-12

    # Calculate the average cross-entropy loss
    data_loss = -np.sum(
        y_onehot * np.log(np.clip(probabilities, eps, 1))
    ) / n

    # Add L2 regularization.
    # The first row contains bias weights, so we exclude it.
    regularization = (
        reg_lambda / (2 * n)
    ) * np.sum(weights[1:, :] ** 2)

    return data_loss + regularization


def softmax_gradient(X, y_onehot, weights, reg_lambda=0.0):
    """
    Calculates the gradient of the Softmax loss.

    The gradient tells us how to update the weights
    to reduce the classification loss.
    """

    # Number of samples
    n = len(y_onehot)

    # Calculate the predicted class probabilities
    probabilities = softmax(X @ weights)

    # Calculate the gradient of cross-entropy loss
    gradient = X.T @ (probabilities - y_onehot) / n

    # Add the L2 regularization gradient to feature weights.
    # Bias weights are excluded.
    gradient[1:, :] += (
        reg_lambda / n
    ) * weights[1:, :]

    return gradient


def train_softmax_regression(
    X, y, learning_rate=0.1,
    epochs=2000, reg_lambda=0.0
):
    """
    Trains the Softmax Regression model using Gradient Descent.

    It learns a separate set of weights for each class.
    """

    # Number of samples and number of features
    n, d = X.shape

    # Find all unique class labels
    classes = np.unique(y)

    # Convert the actual labels into one-hot vectors
    y_onehot = one_hot_encode(y, classes)

    # Add a bias column to the feature matrix
    X_bias = np.column_stack([np.ones(n), X])

    # Create a weight matrix:
    # Rows = features + bias
    # Columns = classes
    weights = np.zeros((d + 1, len(classes)))

    # Save the loss at every epoch
    losses = []

    # Start training
    for epoch in range(epochs):

        # Calculate the gradient using the current weights
        gradient = softmax_gradient(
            X_bias, y_onehot, weights, reg_lambda
        )

        # Update the weights
        weights -= learning_rate * gradient

        # Calculate the new loss after updating the weights
        loss = softmax_loss(
            X_bias, y_onehot, weights, reg_lambda
        )

        losses.append(loss)

    # Return the learned weights, class labels, and loss history
    return weights, classes, losses


def softmax_predict_proba(X, weights):
    """
    Returns the predicted probability of every class for each sample.
    """

    # Add the bias column
    X_bias = np.column_stack([np.ones(len(X)), X])

    # Calculate probabilities using the learned weights
    return softmax(X_bias @ weights)


def softmax_predict(X, weights, classes):
    """
    Predicts the class with the highest probability.

    np.argmax finds the position of the largest probability.
    We use classes to convert that position into the original label.
    """

    # Calculate probabilities for all classes
    probabilities = softmax_predict_proba(X, weights)

    # Find the class with the highest predicted probability
    return classes[np.argmax(probabilities, axis=1)]

In [ ]:
# ==========================================
# PREPARE DATASETS FOR SOFTMAX
# ==========================================

def prepare_softmax_dataset(dataset_name, standardize=True):
    """
    Extracts the train, validation, and test data.

    If standardize=True, it standardizes the features
    using the training data's mean and standard deviation.
    """

    # Get the selected dataset from Person 1's splits dictionary
    data = splits[dataset_name]

    # Copy the feature arrays so we do not accidentally
    # modify the original arrays inside splits.
    X_train = data["X_train"].copy()
    X_val = data["X_val"].copy()
    X_test = data["X_test"].copy()

    # Extract the actual labels
    y_train = data["y_train"]
    y_val = data["y_val"]
    y_test = data["y_test"]

    # Standardize only if requested
    if standardize:

        # Calculate mean and standard deviation from training data
        mean = X_train.mean(axis=0)
        std = X_train.std(axis=0)

        # Prevent division by zero
        std[std == 0] = 1

        # Apply the same transformation to all three splits
        X_train = (X_train - mean) / std
        X_val = (X_val - mean) / std
        X_test = (X_test - mean) / std

    # Return all features and labels
    return X_train, X_val, X_test, y_train, y_val, y_test


# Prepare Iris
# Here we standardize the features using training data statistics.
iris_data = prepare_softmax_dataset(
    "Iris",
    standardize=True
)

# Prepare Digits
# The original dataset-loading code already divided pixel values by 16.
digits_data = prepare_softmax_dataset(
    "Digits",
    standardize=False
)

In [ ]:
# ==========================================
# TRAIN AND EVALUATE SOFTMAX
# ==========================================

# Dictionary to store the results of each dataset
softmax_results = {}


# Train the same model on Iris and Digits
for name, data in [
    ("Iris", iris_data),
    ("Digits", digits_data)
]:

    # Unpack the features and labels
    X_train, X_val, X_test, y_train, y_val, y_test = data

    # Train Softmax Regression using the training data
    weights, classes, losses = train_softmax_regression(
        X_train,
        y_train,
        learning_rate=0.1,
        epochs=2000,
        reg_lambda=0.01
    )

    # Predict labels for the training set
    train_pred = softmax_predict(X_train, weights, classes)

    # Predict labels for the validation set
    val_pred = softmax_predict(X_val, weights, classes)

    # Predict labels for the test set
    test_pred = softmax_predict(X_test, weights, classes)

    # Store the model parameters, loss history, and accuracy results
    softmax_results[name] = {
        "weights": weights,
        "classes": classes,
        "losses": losses,

        # Accuracy is the fraction of correctly classified samples
        "train_accuracy": np.mean(train_pred == y_train),
        "val_accuracy": np.mean(val_pred == y_val),
        "test_accuracy": np.mean(test_pred == y_test)
    }

    # Display the results for the current dataset
    print(f"\n{name}")

    print("Training accuracy:",
          softmax_results[name]["train_accuracy"])

    print("Validation accuracy:",
          softmax_results[name]["val_accuracy"])

    print("Test accuracy:",
          softmax_results[name]["test_accuracy"])

    # Plot the loss to see how the model learned over time
    plt.figure(figsize=(7, 4))

    plt.plot(losses)

    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.title(f"{name} Softmax Training Loss")

    plt.grid(True)
    plt.show()